In [2]:
import torch
from transformers import AutoModel, AutoTokenizer

from conn import DeBERTaEncoder

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
MODEL_NAME = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
encoder = DeBERTaEncoder(model, tokenizer, DEVICE)

print("device:", DEVICE)
print("model:", MODEL_NAME)

/opt/miniconda3/envs/connections/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1754.33it/s, Materializing param=encoder.rel_embeddings.weight]                   
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mas

device: mps
model: microsoft/deberta-v3-small


In [3]:
# Parameters (rank=8 as chosen ideal from experiment)
N_TRAIN = 1000
N_EVAL = 1000
EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
TEMPERATURE = 0.07
ADAPTER_DIR = "adapters/deberta-lora-r8-final"
VERBOSE = False
SAVE_RESULTS = True

In [4]:
from data_loader import get_train_test_split, gold_example_groups_from_row

ds_train, ds_test = get_train_test_split()
hf_split = ds_train
test_split = ds_test
train_puzzles = [hf_split[i] for i in range(min(N_TRAIN, len(hf_split)))]
print("train puzzles (for fine-tuning):", len(train_puzzles))
print("full train split:", len(hf_split))
print("test split:", len(test_split))

train puzzles (for fine-tuning): 521
full train split: 521
test split: 131


In [5]:
from conn import finetune_deberta_lora

print("Starting fine-tuning (LoRA rank=8)...")
ft_encoder, stats = finetune_deberta_lora(
    puzzles=train_puzzles,
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LR,
    temperature=TEMPERATURE,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    adapter_output_dir=ADAPTER_DIR,
    verbose=True,
)
print(f"Training complete: {stats.epochs} epochs, {stats.steps} steps")
print(f"Final avg loss: {stats.average_losses[-1]:.4f}")

Starting fine-tuning (LoRA rank=8)...
Starting contextual LoRA fine-tuning on mps for 3 epochs...


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1461.75it/s, Materializing param=encoder.rel_embeddings.weight]                   
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignor

Epoch 1/3 | Average Loss: 3.1898
Epoch 2/3 | Average Loss: 2.7949
Epoch 3/3 | Average Loss: 2.7574
Training complete: 3 epochs, 393 steps
Final avg loss: 2.7574


In [6]:
from conn import load_lora_encoder

loaded_encoder = load_lora_encoder(ADAPTER_DIR, base_model_name=MODEL_NAME)
print(f"Loaded LoRA encoder from {ADAPTER_DIR}")

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1075.37it/s, Materializing param=encoder.rel_embeddings.weight]                   
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignor

Loaded LoRA encoder from adapters/deberta-lora-r8-final


In [7]:
from conn.solvers import FewShotSolver

solver = FewShotSolver(loaded_encoder)


def solve_puzzle(words16):
    return solver.solve(words16)

In [8]:
import json
import time
from datetime import datetime
from pathlib import Path

from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    correct_word_count,
    evaluate,
    n_correct_groups,
)
from data_loader import gold_example_groups_from_row


def gold_words_from_row(row):
    return [eg.words for eg in gold_example_groups_from_row(row)]


def _build_output_row(i, row, words16, gold_egs, gold, pred, z, s, inference_time_sec=0.0):
    levels = [eg.level for eg in gold_egs]
    valid = len(pred) == 4 and all(len(g) == 4 for g in pred)
    if not valid:
        return None
    return {
        "index": i,
        "words16": words16,
        "pred_groups": pred,
        "gold_groups": gold,
        "zero_one": z,
        "min_swaps": s,
        "date": str(row.get("date", "")),
        "levels": levels,
        "n_correct_groups": n_correct_groups(pred, gold),
        "correct_word_count": correct_word_count(pred, gold),
        "inference_time_sec": inference_time_sec,
        "is_valid": True,
    }


if SAVE_RESULTS:
    run_dir = Path("results") / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    n = min(N_EVAL, len(hf_split))
    for i in range(n):
        row = hf_split[i]
        words16 = row.get("words", [])
        if len(words16) != 16:
            continue
        gold_egs = gold_example_groups_from_row(row)
        if len(gold_egs) != 4:
            continue
        gold = [eg.words for eg in gold_egs]
        t0 = time.perf_counter()
        pred = solve_puzzle(words16)
        inference_sec = time.perf_counter() - t0
        z = accuracy_zero_one(pred, gold)
        s = accuracy_min_swaps(pred, gold)
        rec = _build_output_row(i, row, words16, gold_egs, gold, pred, z, s, inference_sec)
        if rec is not None:
            outputs.append(rec)
    if outputs:
        acc = sum(o["zero_one"] for o in outputs) / len(outputs)
        mean_swaps = sum(o["min_swaps"] for o in outputs) / len(outputs)
        n_eval = len(outputs)
        n_cg = sum(o["n_correct_groups"] for o in outputs) / len(outputs)
        cwc = sum(o["correct_word_count"] for o in outputs) / len(outputs)
        mean_inference_sec = sum(o["inference_time_sec"] for o in outputs) / len(outputs)
        total_inference_sec = sum(o["inference_time_sec"] for o in outputs)
        metrics = {
            "split": "train",
            "zero_one_accuracy": acc,
            "mean_min_swaps": mean_swaps,
            "mean_n_correct_groups": n_cg,
            "mean_correct_word_count": cwc,
            "mean_inference_time_sec": mean_inference_sec,
            "total_inference_time_sec": total_inference_sec,
            "n_eval": n_eval,
        }
        (run_dir / "deberta_lora_metrics.json").write_text(json.dumps(metrics, indent=2))
        (run_dir / "deberta_lora_outputs.json").write_text(json.dumps(outputs, indent=2))
        print(f"Saved train results to {run_dir}/ (metrics + outputs)")
    else:
        acc, n_eval, mean_swaps = 0.0, 0, 0.0
else:
    results = evaluate(
        hf_split,
        metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
        solver_fn=solve_puzzle,
        max_samples=N_EVAL,
        gold_from_row=gold_words_from_row,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

print(f"Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
print(f"Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")

Saved train results to results/2026-03-15_01-34-09/ (metrics + outputs)
Zero-one accuracy: 0.0058  (n=521, requested=1000)
Mean 1-1 swaps to correct: 3.42  (n=521)


## Run on test split (saves to results/ when SAVE_RESULTS is True)

Run this cell to evaluate on the held-out test split. Results saved to `deberta_lora_test_metrics.json` and `deberta_lora_test_outputs.json` in the same timestamped run folder.

In [9]:
if SAVE_RESULTS:
    import time
    from datetime import datetime
    run_dir = globals().get("run_dir") or (Path("results") / datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    outputs_test = []
    n_test = min(N_EVAL, len(test_split))
    for i in range(n_test):
        row = test_split[i]
        words16 = row.get("words", [])
        if len(words16) != 16:
            continue
        gold_egs = gold_example_groups_from_row(row)
        if len(gold_egs) != 4:
            continue
        gold = [eg.words for eg in gold_egs]
        t0 = time.perf_counter()
        pred = solve_puzzle(words16)
        inference_sec = time.perf_counter() - t0
        z = accuracy_zero_one(pred, gold)
        s = accuracy_min_swaps(pred, gold)
        rec = _build_output_row(i, row, words16, gold_egs, gold, pred, z, s, inference_sec)
        if rec is not None:
            outputs_test.append(rec)
    if outputs_test:
        acc_t = sum(o["zero_one"] for o in outputs_test) / len(outputs_test)
        mean_swaps_t = sum(o["min_swaps"] for o in outputs_test) / len(outputs_test)
        n_cg_t = sum(o["n_correct_groups"] for o in outputs_test) / len(outputs_test)
        cwc_t = sum(o["correct_word_count"] for o in outputs_test) / len(outputs_test)
        mean_inference_sec_t = sum(o["inference_time_sec"] for o in outputs_test) / len(outputs_test)
        total_inference_sec_t = sum(o["inference_time_sec"] for o in outputs_test)
        metrics_test = {
            "split": "test",
            "zero_one_accuracy": acc_t,
            "mean_min_swaps": mean_swaps_t,
            "mean_n_correct_groups": n_cg_t,
            "mean_correct_word_count": cwc_t,
            "mean_inference_time_sec": mean_inference_sec_t,
            "total_inference_time_sec": total_inference_sec_t,
            "n_eval": len(outputs_test),
        }
        (run_dir / "deberta_lora_test_metrics.json").write_text(json.dumps(metrics_test, indent=2))
        (run_dir / "deberta_lora_test_outputs.json").write_text(json.dumps(outputs_test, indent=2))
        print(f"Saved test results to {run_dir}/ (deberta_lora_test_*)")
        print(f"Test zero-one: {acc_t:.4f}  mean min_swaps: {mean_swaps_t:.2f}  n={len(outputs_test)}")
    else:
        print("No valid test outputs to save.")
else:
    results_test = evaluate(
        test_split,
        metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
        solver_fn=solve_puzzle,
        max_samples=N_EVAL,
        gold_from_row=gold_words_from_row,
    )
    acc_t, n_t = results_test["zero_one"]
    mean_swaps_t, _ = results_test["min_swaps"]
    print(f"Test zero-one accuracy: {acc_t:.4f}  (n={n_t})")
    print(f"Test mean min_swaps: {mean_swaps_t:.2f}")

Saved test results to results/2026-03-15_01-34-09/ (deberta_lora_test_*)
Test zero-one: 0.0000  mean min_swaps: 3.44  n=131


## Summarize saved results

Loads the latest timestamped run folder that contains LoRA metrics and prints train/test summary.

In [10]:
results_base = Path("results")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], reverse=True)
run_dir = next((d for d in run_dirs if (d / "deberta_lora_metrics.json").exists()), None)
if run_dir is None:
    print("No timestamped run found with deberta_lora_metrics.json")
else:
    print(f"Run folder: {run_dir}\n")
    for name, path in [
        ("LoRA (train)", "deberta_lora_metrics.json"),
        ("LoRA (test)", "deberta_lora_test_metrics.json"),
    ]:
        p = run_dir / path
        if p.exists():
            m = json.loads(p.read_text())
            print(f"{name}:")
            for k, v in m.items():
                if isinstance(v, float):
                    print(f"  {k}: {v:.4f}")
                else:
                    print(f"  {k}: {v}")
            print()
        else:
            print(f"{name}: (file not found: {path})\n")

Run folder: results/2026-03-15_01-34-09

LoRA (train):
  split: train
  zero_one_accuracy: 0.0058
  mean_min_swaps: 3.4184
  mean_n_correct_groups: 0.3666
  mean_correct_word_count: 9.6718
  mean_inference_time_sec: 0.1933
  total_inference_time_sec: 100.6883
  n_eval: 521

LoRA (test):
  split: test
  zero_one_accuracy: 0.0000
  mean_min_swaps: 3.4351
  mean_n_correct_groups: 0.3740
  mean_correct_word_count: 9.6565
  mean_inference_time_sec: 0.1860
  total_inference_time_sec: 24.3687
  n_eval: 131

